### Libraries

In [31]:
# to install packages if not installed
# %pip install pandas
# %pip install seaborn
# %pip install statsmodels
# %pip install matplotlib
# %pip install numpy

In [32]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np

In [33]:
df = pd.read_excel("data/Data_FE.xlsx", sheet_name="25 Size and BEME portfolios")
df.head()

,Unnamed: 0,Average,Value,Weighted,Returns,--,Monthly,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,Size,Small,Small,Small,Small,Small,2,2.00,2.00,2.00,...,4,4.00,4.00,4.00,4,Big,Big,Big,Big,Big
1,BE/ME,Low,2,3,4,High,Low,2.00,3.00,4.00,...,Low,2.00,3.00,4.00,High,Low,2,3,4,High
2,193601,26.94,15.49,21.82,14.56,33.28,16.43,12.49,9.49,10.27,...,2.13,6.87,8.92,8.35,6.17,2.55,6.36,8.62,14.84,16.29
3,193602,9.46,12.78,6.77,10.02,9,1.7,6.71,5.61,8.92,...,2.59,4.04,6.14,4.80,8.43,1.88,1.18,3.99,3.38,3.75
4,193603,9.46,1.38,5.56,3.01,1.24,-0.37,1.35,3.33,-1.11,...,-0.46,-0.40,3.19,1.52,-4.36,3.58,1.27,-1.82,1.26,-3.56


In [34]:
# Set rows 1 and 2 as MultiIndex header
df.columns = pd.MultiIndex.from_arrays([df.iloc[0], df.iloc[1]])
df = df.iloc[2:].reset_index(drop=True)

# Convert Date column to Year-Month
date_col = df.columns[0]

df[date_col] = pd.to_datetime(df[date_col].astype(int), format="%Y%m")

df = df.set_index(df.columns[0]) 
df = df.apply(pd.to_numeric)

df.head()

0              Small                                  2                              ...     4                              Big                          
1                Low      2      3      4   High    Low      2      3      4   High  ...   Low     2      3      4   High   Low     2     3      4   High
(Size, BE/ME)                                                                        ...                                                                 
1936-01-01     26.94  15.49  21.82  14.56  33.28  16.43  12.49   9.49  10.27  26.81  ...  2.13  6.87   8.92   8.35   6.17  2.55  6.36  8.62  14.84  16.29
1936-02-01      9.46  12.78   6.77  10.02   9.00   1.70   6.71   5.61   8.92   3.87  ...  2.59  4.04   6.14   4.80   8.43  1.88  1.18  3.99   3.38   3.75
1936-03-01      9.46   1.38   5.56   3.01   1.24  -0.37   1.35   3.33  -1.11   1.25  ... -0.46 -0.40   3.19   1.52  -4.36  3.58  1.27 -1.82   1.26  -3.56
1936-04-01    -28.60 -29.05 -12.37 -14.03 -21.72 -19.41 -13.35 -15.93 -16.43 -17.69  ... -8.31 -9.00 -11.90 -12.09 -12.46 -6.17 -8.26 -7.05 -10.57  -8.00
1936-05-01      1.81   8.62   1.89  10.60   5.90   5.21   4.59   7.57   5.84   6.97  ...  5.12  1.79   4.54   5.98   8.18  4.78  5.20  4.71   6.25   8.39

[5 rows x 25 columns]

In [35]:
factors = pd.read_excel("data/Data_FE.xlsx", sheet_name="Fama-French factors")

# Convert Date column to Year-Month
date_col = factors.columns[0] 

factors[date_col] = pd.to_datetime(factors[date_col].astype(int), format="%Y%m")
factors = factors.set_index(factors.columns[0]) 
factors = factors.apply(pd.to_numeric)

factors.head()

,Mkt-RF,SMB,HML,RF
Date,,,,
1936-01-01,6.60,6.43,10.09,0.01
1936-02-01,2.56,0.77,3.98,0.01
1936-03-01,0.92,1.10,-2.23,0.02
1936-04-01,-8.07,-6.81,-2.18,0.02
1936-05-01,5.01,0.81,2.69,0.02


In [36]:
target_df = df.copy()

factors = factors.reindex(target_df.index)

target_df = target_df.sub(factors["RF"], axis=0)
target_df.head()

0              Small                                  2                              ...     4                              Big                          
1                Low      2      3      4   High    Low      2      3      4   High  ...   Low     2      3      4   High   Low     2     3      4   High
(Size, BE/ME)                                                                        ...                                                                 
1936-01-01     26.93  15.48  21.81  14.55  33.27  16.42  12.48   9.48  10.26  26.80  ...  2.12  6.86   8.91   8.34   6.16  2.54  6.35  8.61  14.83  16.28
1936-02-01      9.45  12.77   6.76  10.01   8.99   1.69   6.70   5.60   8.91   3.86  ...  2.58  4.03   6.13   4.79   8.42  1.87  1.17  3.98   3.37   3.74
1936-03-01      9.44   1.36   5.54   2.99   1.22  -0.39   1.33   3.31  -1.13   1.23  ... -0.48 -0.42   3.17   1.50  -4.38  3.56  1.25 -1.84   1.24  -3.58
1936-04-01    -28.62 -29.07 -12.39 -14.05 -21.74 -19.43 -13.37 -15.95 -16.45 -17.71  ... -8.33 -9.02 -11.92 -12.11 -12.48 -6.19 -8.28 -7.07 -10.59  -8.02
1936-05-01      1.79   8.60   1.87  10.58   5.88   5.19   4.57   7.55   5.82   6.95  ...  5.10  1.77   4.52   5.96   8.16  4.76  5.18  4.69   6.23   8.37

[5 rows x 25 columns]

## Fama-MacBeth Corss-sectional Approach

$$r_{it}=a_i+\beta_{1i}proxy_t^{size}+\beta_{2i}proxy_t^{beme}+\beta_{3i}Market_t+\epsilon_{it}$$

### Linear Regression using Statsmodels package

In [37]:
X = factors[["Mkt-RF", "SMB", "HML"]]
X = sm.add_constant(X)
y = target_df["Small"]["Low"]


model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    Low   R-squared:                       0.826
Model:                            OLS   Adj. R-squared:                  0.826
Method:                 Least Squares   F-statistic:                     1372.
Date:                Sun, 22 Mar 2026   Prob (F-statistic):               0.00
Time:                        13:33:56   Log-Likelihood:                -2391.7
No. Observations:                 870   AIC:                             4791.
Df Residuals:                     866   BIC:                             4811.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.7237      0.131     -5.505      0.0

In [38]:
X = factors[["Mkt-RF", "SMB", "HML"]]
X = sm.add_constant(X)

r_squa = []
const = []
Mkt_Rf = []
SMB = []
HML = []

for j in range(len(target_df.columns)):
    y = target_df.iloc[:, j]
    model = sm.OLS(y, X).fit()
    r_squa.append(model.rsquared)
    parameters = model.params
    const.append(parameters.iloc[0])
    Mkt_Rf.append(parameters.iloc[1])
    SMB.append(parameters.iloc[2])
    HML.append(parameters.iloc[3])


dat = {"R-squared": r_squa, "Constant": const, "MKT-Rf" : Mkt_Rf, "SMB" : SMB, "HML" : HML}

summary_df = pd.DataFrame(dat, index=target_df.columns)
print(summary_df)


            R-squared  Constant    MKT-Rf       SMB       HML
0     1                                                      
Small Low    0.826192 -0.723700  1.177556  1.602898  0.058750
      2      0.890382 -0.239201  1.066069  1.331507  0.204666
      3      0.903950 -0.137339  1.018885  1.207639  0.475800
      4      0.939661  0.060985  0.948844  1.075341  0.537487
      High   0.927522 -0.004267  1.024591  1.252761  0.927741
2     Low    0.921889 -0.262442  1.146522  1.095769 -0.192022
      2      0.937168 -0.019206  1.022803  0.898455  0.137733
      3      0.938942  0.086304  0.950778  0.758805  0.317551
      4      0.945900  0.058399  0.973322  0.729197  0.523356
      High   0.954548 -0.032821  1.098261  0.885679  0.836702
3     Low    0.943537 -0.061043  1.096843  0.758059 -0.370269
      2      0.913399  0.051431  1.025852  0.513394  0.109341
      3      0.910198  0.043816  0.994081  0.441875  0.367561
      4      0.910629  0.087798  0.960576  0.385591  0.519513
      Hi

## Macroeconomics Factor Model

In [39]:
macro_factors = pd.read_excel("data/Data_FE.xlsx", sheet_name="Macroeconomic factors")

# Convert Date column to Year-Month
date_col = macro_factors.columns[0] 

macro_factors[date_col] = pd.to_datetime(macro_factors[date_col].astype(int), format="%Y%m")
macro_factors = macro_factors.set_index(macro_factors.columns[0]) 
macro_factors = macro_factors.apply(pd.to_numeric)

macro_factors.head()

,Div_growth,DEF = LT Corp. - LT govt.,TERM = ST govt.-LT govt.
Date,,,
1936-01-01,0.001990,0.002660,0.005395
1936-02-01,0.003113,-0.002690,0.007990
1936-03-01,0.003308,-0.002406,0.010458
1936-04-01,0.001250,-0.000924,0.003359
1936-05-01,0.004499,-0.000049,0.003887


## In-sample Principle Component Factors

In [40]:
# excess returns
excess_returns = df.copy()

macro_factors = macro_factors.reindex(excess_returns.index)
excess_returns = excess_returns.sub(macro_factors["TERM = ST govt.-LT govt."], axis=0)
excess_returns.head()

# covariance matrix of excess returns
cova_matrix = excess_returns.cov()

# Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(cova_matrix)

# eigh returns in ASCENDING order — reverse to get largest first
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]    # 25 × 25, columns are eigenvectors

# extracting 3 largest eigenvectors, the factor weights is 25 x 3
W = eigenvectors[:, :3]

# factor time-series (T x 3)
F = excess_returns.values @ W 

in_sample_factors_df = pd.DataFrame(F, index=excess_returns.index, columns=['In-sample-PC1', 'In-sample-PC2', 'In-sample-PC3'])

in_sample_factors_df.head()

,In-sample-PC1,In-sample-PC2,In-sample-PC3
"(Size, BE/ME)",,,
1936-01-01,68.032285,-8.331803,20.995633
1936-02-01,27.507629,-1.799596,7.564300
1936-03-01,8.558362,-6.086865,-1.359546
1936-04-01,-70.181755,11.244138,-4.735477
1936-05-01,27.877746,7.509749,4.294490


### Linear regression for In-sample PC factors

In [41]:
X = in_sample_factors_df[["In-sample-PC1", "In-sample-PC2", "In-sample-PC3"]]
X = sm.add_constant(X)

r_squa = []
const = []
In_sample_PC1 = []
In_sample_PC2 = []
In_sample_PC3 = []

for j in range(len(df.columns)):
    y = df.iloc[:, j]
    model = sm.OLS(y, X).fit()
    r_squa.append(model.rsquared)
    parameters = model.params
    const.append(parameters.iloc[0])
    In_sample_PC1.append(parameters.iloc[1])
    In_sample_PC2.append(parameters.iloc[2])
    In_sample_PC3.append(parameters.iloc[3])


dat = {"R-squared": r_squa, "Constant": const, "In-sample-PC1" : In_sample_PC1, "In-sample-PC2" : In_sample_PC2, "In-sample-PC3" : In_sample_PC3}

summary_In_sample = pd.DataFrame(dat, index=df.columns)
print(summary_In_sample)

            R-squared  Constant  In-sample-PC1  In-sample-PC2  In-sample-PC3
0     1                                                                     
Small Low    0.921715 -0.396936       0.285857      -0.493297      -0.007568
      2      0.919575 -0.010668       0.249565      -0.290141       0.024344
      3      0.936452  0.061332       0.237898      -0.196171       0.160877
      4      0.951515  0.252369       0.217832      -0.131189       0.173694
      High   0.956433  0.213742       0.244886      -0.139502       0.381305
2     Low    0.943661 -0.143758       0.242710      -0.222455      -0.234801
      2      0.941641  0.067974       0.213728      -0.077401      -0.082937
      3      0.938897  0.175776       0.194900      -0.007690      -0.000744
      4      0.939469  0.148597       0.197809       0.034764       0.080957
      High   0.947824  0.055243       0.229625       0.040224       0.231439
3     Low    0.954426  0.008199       0.209374      -0.125585      -0.355949

## Out-sample principle Component Factors

### Even and Odd Monthly Returns dataframes

In [42]:
# making odd monthly return and even monthly returns dataframe
odd_returns_df = df.copy()
odd_returns_df = odd_returns_df[::1]

even_returns_df = df.copy()
even_returns_df = even_returns_df[::1]

# covariance matrix
cova_matrix_even = even_returns_df.cov()
cova_matrix_odd = odd_returns_df.cov()

In [43]:
# Eigendecomposition for even
eigenvalues_even, eigenvectors_even = np.linalg.eigh(cova_matrix_even)

# eigh returns in ascending order — reverse to get largest first
idx = np.argsort(eigenvalues_even)[::-1]
eigenvalues_even  = eigenvalues_even[idx]
eigenvectors_even = eigenvectors_even[:, idx]    # 25 × 25, columns are eigenvectors

# extracting 3 largest eigenvectors, the factor weights is 25 x 3
W_even = eigenvectors_even[:, :3]

# Eigendecomposition for odd
eigenvalues_odd, eigenvectors_odd = np.linalg.eigh(cova_matrix_odd)

# eigh returns in ascending order — reverse to get largest first
idx = np.argsort(eigenvalues_odd)[::-1]
eigenvalues_odd  = eigenvalues_odd[idx]
eigenvectors_odd = eigenvectors_odd[:, idx]    # 25 × 25, columns are eigenvectors

# extracting 3 largest eigenvectors, the factor weights is 25 x 3
W_odd = eigenvectors_odd[:, :3]

# factor time-series (T x 25)
F_even = even_returns_df.values @ W_odd
F_odd = odd_returns_df.values @ W_even

# Create empty dataframe with full index
out_sample_factors_df = pd.DataFrame(index=df.index, columns=['Out-sample-PC1', 'Out-sample-PC2', 'Out-sample-PC3'], dtype=float)

# Assign values to correct timestamps
out_sample_factors_df.loc[even_returns_df.index] = F_even
out_sample_factors_df.loc[odd_returns_df.index]  = F_odd

out_sample_factors_df.head()

,Out-sample-PC1,Out-sample-PC2,Out-sample-PC3
"(Size, BE/ME)",,,
1936-01-01,68.053580,-8.384532,20.986663
1936-02-01,27.545814,-1.812116,7.559540
1936-03-01,8.608734,-6.075599,-1.375887
1936-04-01,-70.161977,11.272641,-4.723863
1936-05-01,27.898146,7.497325,4.310874


### Linear regression for Out-sample PC factors

In [44]:
X = out_sample_factors_df[["Out-sample-PC1", "Out-sample-PC2", "Out-sample-PC3"]]
X = sm.add_constant(X)

r_squa = []
const = []
out_sample_PC1 = []
out_sample_PC2 = []
out_sample_PC3 = []

for j in range(len(df.columns)):
    y = df.iloc[:, j]
    model = sm.OLS(y, X).fit()
    r_squa.append(model.rsquared)
    parameters = model.params
    const.append(parameters.iloc[0])
    out_sample_PC1.append(parameters.iloc[1])
    out_sample_PC2.append(parameters.iloc[2])
    out_sample_PC3.append(parameters.iloc[3])


dat = {"R-squared": r_squa, "Constant": const, "Out-sample-PC1" : out_sample_PC1, "Out-sample-PC2" : out_sample_PC2, "Out-sample-PC3" : out_sample_PC3}

pd.set_option('display.width', 500)
summary_out_sample = pd.DataFrame(dat, index=df.columns)
print(summary_out_sample)

            R-squared  Constant  Out-sample-PC1  Out-sample-PC2  Out-sample-PC3
0     1                                                                        
Small Low    0.921776 -0.397347        0.285651       -0.493919       -0.008052
      2      0.919559 -0.010926        0.249392       -0.290861        0.024271
      3      0.936501  0.060984        0.237747       -0.197081        0.160933
      4      0.951516  0.252098        0.217693       -0.132108        0.173861
      High   0.956379  0.213573        0.244702       -0.140883        0.381507
2     Low    0.943552 -0.143955        0.242563       -0.222825       -0.234642
      2      0.941608  0.067682        0.213620       -0.077944       -0.082589
      3      0.938990  0.175384        0.194820       -0.008291       -0.000336
      4      0.939590  0.148172        0.197730        0.034011        0.081442
      High   0.947795  0.054954        0.229502        0.039055        0.232032
3     Low    0.954348  0.007920        0